In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Thu Aug 27 06:52:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.3 MB/s eta 0:00:00:00:01


In [3]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bnb.__version__)

print()
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

Torch: 2.10.0+cu128
Transformers: 5.16.1
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.12.0
bitsandbytes: 0.50.2

CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded")
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded
Vocab size: 151643


In [5]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")
print("Parameters:", model.num_parameters())
print("Device map:", model.hf_device_map)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully
Parameters: 7615616512
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [6]:
messages = [
    {
        "role": "user",
        "content": "Explain what an event horizon is in simple scientific terms."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

An event horizon is a boundary around a black hole beyond which nothing can escape, including light. To understand this concept more simply:

Imagine you're throwing a ball into a deep, dark well. If the well is deep enough, once the ball passes a certain point, it will be pulled so strongly by the well's gravity that it can never come back out. The point from which the ball cannot return is like the "edge" of the well, and for a black hole, this edge is


In [7]:
from pathlib import Path

for directory in [
    "fine_tuning_lab",
    "fine_tuning_lab/data",
    "fine_tuning_lab/checkpoints",
    "fine_tuning_lab/results",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Experiment structure created")


Experiment structure created


In [10]:
import requests
import pandas as pd

url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

query = """
SELECT
    pl_name,
    hostname,
    discoverymethod,
    disc_year,
    pl_orbper,
    pl_rade,
    pl_masse,
    pl_eqt,
    st_teff,
    st_mass,
    st_rad,
    sy_dist
FROM ps
WHERE
    pl_name IS NOT NULL
    AND pl_orbper IS NOT NULL
    AND st_mass IS NOT NULL
"""

response = requests.get(
    url,
    params={
        "query": query,
        "format": "json",
    },
    timeout=60,
)

response.raise_for_status()

df = pd.DataFrame(response.json())

print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())

Rows: 32483
Columns: ['pl_name', 'hostname', 'discoverymethod', 'disc_year', 'pl_orbper', 'pl_rade', 'pl_masse', 'pl_eqt', 'st_teff', 'st_mass', 'st_rad', 'sy_dist']


,pl_name,hostname,discoverymethod,disc_year,pl_orbper,pl_rade,pl_masse,pl_eqt,st_teff,st_mass,st_rad,sy_dist
0,Kepler-317 c,Kepler-317,Transit,2014,8.775000,NaN,NaN,NaN,NaN,0.980,0.93600,940.584
1,Kepler-1513 b,Kepler-1513,Transit,2016,160.884200,8.594,48.309918,NaN,5491.00,0.943,0.95000,349.247
2,HAT-P-45 b,HAT-P-45,Transit,2014,3.128995,NaN,NaN,NaN,6330.00,1.259,1.31900,298.640
3,NGTS-1 b,NGTS-1,Transit,2017,2.647305,NaN,NaN,NaN,3916.00,0.617,0.57300,218.121
4,Kepler-1181 b,Kepler-1181,Transit,2016,4.893433,NaN,NaN,NaN,6495.28,1.330,1.36041,938.417


In [11]:
required = [
    "pl_name",
    "hostname",
    "discoverymethod",
    "disc_year",
    "pl_orbper",
    "st_mass",
]

clean = df.dropna(subset=required).copy()

clean = clean.drop_duplicates(
    subset=["pl_name"]
)

print("Raw rows:", len(df))
print("Clean rows:", len(clean))

display(clean.head())

Raw rows: 32483
Clean rows: 5877


,pl_name,hostname,discoverymethod,disc_year,pl_orbper,pl_rade,pl_masse,pl_eqt,st_teff,st_mass,st_rad,sy_dist
0,Kepler-317 c,Kepler-317,Transit,2014,8.775000,NaN,NaN,NaN,NaN,0.980,0.93600,940.584
1,Kepler-1513 b,Kepler-1513,Transit,2016,160.884200,8.594,48.309918,NaN,5491.00,0.943,0.95000,349.247
2,HAT-P-45 b,HAT-P-45,Transit,2014,3.128995,NaN,NaN,NaN,6330.00,1.259,1.31900,298.640
3,NGTS-1 b,NGTS-1,Transit,2017,2.647305,NaN,NaN,NaN,3916.00,0.617,0.57300,218.121
4,Kepler-1181 b,Kepler-1181,Transit,2016,4.893433,NaN,NaN,NaN,6495.28,1.330,1.36041,938.417


In [12]:
import numpy as np

clean["period_years"] = clean["pl_orbper"] / 365.25

clean["semi_major_axis_au"] = (
    clean["st_mass"] *
    clean["period_years"] ** 2
) ** (1 / 3)

display(
    clean[
        [
            "pl_name",
            "st_mass",
            "pl_orbper",
            "semi_major_axis_au",
        ]
    ].head(10)
)

,pl_name,st_mass,pl_orbper,semi_major_axis_au
0,Kepler-317 c,0.980,8.775000,0.082701
1,Kepler-1513 b,0.943,160.884200,0.567701
2,HAT-P-45 b,1.259,3.128995,0.045208
3,NGTS-1 b,0.617,2.647305,0.031884
4,Kepler-1181 b,1.330,4.893433,0.062034
5,Kepler-26 d,0.550,3.543918,0.037272
6,Kepler-387 b,1.038,6.791648,0.071065
7,WASP-75 b,1.180,2.484190,0.037934
8,TOI-4914 b,1.030,10.600570,0.095376
9,Kepler-1408 b,1.120,2.997927,0.042256


In [13]:
def make_orbit_example(row):
    star = row["hostname"]
    planet = row["pl_name"]

    mass = float(row["st_mass"])
    period_days = float(row["pl_orbper"])
    period_years = period_days / 365.25
    semi_major_axis = (mass * period_years**2) ** (1 / 3)

    question = (
        f"{planet} orbits the star {star}. "
        f"The host star has a mass of {mass:.3f} solar masses, "
        f"and the planet has an orbital period of "
        f"{period_days:.3f} days. "
        f"Using Kepler's third law, approximately how far "
        f"is the planet from its star in AU?"
    )

    answer = (
        f"Using Kepler's third law, the estimated "
        f"semi-major axis is approximately "
        f"{semi_major_axis:.3f} AU. "
        f"This estimate assumes the stellar mass is given "
        f"in solar masses and the orbital period in years."
    )

    return {
        "messages": [
            {
                "role": "user",
                "content": question,
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ],
        "metadata": {
            "source": "NASA Exoplanet Archive",
            "task": "orbital_reasoning",
            "planet": planet,
            "host_star": star,
        },
    }


examples = [
    make_orbit_example(row)
    for _, row in clean.head(20).iterrows()
]

print(examples[0])

{'messages': [{'role': 'user', 'content': "Kepler-317 c orbits the star Kepler-317. The host star has a mass of 0.980 solar masses, and the planet has an orbital period of 8.775 days. Using Kepler's third law, approximately how far is the planet from its star in AU?"}, {'role': 'assistant', 'content': "Using Kepler's third law, the estimated semi-major axis is approximately 0.083 AU. This estimate assumes the stellar mass is given in solar masses and the orbital period in years."}], 'metadata': {'source': 'NASA Exoplanet Archive', 'task': 'orbital_reasoning', 'planet': 'Kepler-317 c', 'host_star': 'Kepler-317'}}


In [15]:
import requests
import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

# ============================================================
# CONFIG
# ============================================================

OUT = Path("fine_tuning_lab/data/astronomy_ft_v1")
OUT.mkdir(parents=True, exist_ok=True)

API_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

# ============================================================
# 1. DOWNLOAD SOURCE DATA
# ============================================================

QUERY = """
SELECT
    pl_name,
    hostname,
    discoverymethod,
    disc_year,
    pl_orbper,
    pl_rade,
    pl_masse,
    pl_eqt,
    st_teff,
    st_mass,
    st_rad,
    sy_dist
FROM ps
WHERE
    pl_name IS NOT NULL
    AND hostname IS NOT NULL
    AND pl_orbper IS NOT NULL
    AND st_mass IS NOT NULL
"""

response = requests.get(
    API_URL,
    params={
        "query": QUERY,
        "format": "json",
    },
    timeout=120,
)

response.raise_for_status()

raw = pd.DataFrame(response.json())

print("Downloaded:", len(raw), "rows")

# ============================================================
# 2. CLEAN
# ============================================================

required = [
    "pl_name",
    "hostname",
    "discoverymethod",
    "disc_year",
    "pl_orbper",
    "st_mass",
]

df = raw.dropna(subset=required).copy()

df = df.drop_duplicates(
    subset=["pl_name"]
).reset_index(drop=True)

# sensible physical bounds
df = df[
    (df["pl_orbper"] > 0) &
    (df["st_mass"] > 0)
].copy()

# ============================================================
# 3. DERIVED SCIENTIFIC FEATURES
# ============================================================

df["period_years"] = df["pl_orbper"] / 365.25

df["semi_major_axis_au"] = (
    df["st_mass"] *
    df["period_years"] ** 2
) ** (1 / 3)

# Planet density when mass + radius exist.
# Earth density = 5.514 g/cm^3.
df["density_earth_ratio"] = np.nan

mask = (
    df["pl_masse"].notna() &
    df["pl_rade"].notna() &
    (df["pl_rade"] > 0)
)

df.loc[mask, "density_earth_ratio"] = (
    df.loc[mask, "pl_masse"] /
    (df.loc[mask, "pl_rade"] ** 3)
)

# ============================================================
# 4. HELPERS
# ============================================================

def clean_number(value, digits=3):
    return f"{float(value):.{digits}f}"


def make_example(
    task,
    question,
    answer,
    row,
):
    return {
        "messages": [
            {
                "role": "user",
                "content": question,
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ],
        "metadata": {
            "source": "NASA Exoplanet Archive",
            "source_table": "ps",
            "task": task,
            "planet": row["pl_name"],
            "host_star": row["hostname"],
            "discovery_method": row["discoverymethod"],
        },
    }


examples = []

# ============================================================
# 5. TASK 1 — ORBITAL REASONING
# ============================================================

for _, row in df.iterrows():

    mass = float(row["st_mass"])
    period = float(row["pl_orbper"])
    axis = float(row["semi_major_axis_au"])

    question = (
        f"{row['pl_name']} orbits {row['hostname']}. "
        f"The host star has a mass of {mass:.3f} solar masses "
        f"and the planet has an orbital period of {period:.3f} days. "
        f"Using Kepler's third law, approximately how far is "
        f"the planet from its star in AU?"
    )

    answer = (
        f"The estimated semi-major axis is approximately "
        f"{axis:.3f} AU. "
        f"This estimate uses the stellar mass in solar masses "
        f"and the orbital period converted to years."
    )

    examples.append(
        make_example(
            "orbital_reasoning",
            question,
            answer,
            row,
        )
    )

# ============================================================
# 6. TASK 2 — OBSERVATIONAL INTERPRETATION
# ============================================================

for _, row in df.iterrows():

    question = (
        f"{row['pl_name']} was discovered using the "
        f"{row['discoverymethod']} method in {int(row['disc_year'])}. "
        f"What does the discovery method tell us about how "
        f"this exoplanet was detected?"
    )

    method = str(row["discoverymethod"])

    if method.lower() == "transit":
        explanation = (
            "The transit method detects a planet when it passes "
            "in front of its host star from our viewpoint, producing "
            "a measurable decrease in the star's observed brightness."
        )
    elif method.lower() == "radial velocity":
        explanation = (
            "The radial-velocity method detects the gravitational "
            "influence of an orbiting planet through changes in the "
            "host star's measured line-of-sight velocity."
        )
    else:
        explanation = (
            f"The planet was identified using the {method} method. "
            f"The method indicates the observational technique used "
            f"to establish evidence for the planet."
        )

    examples.append(
        make_example(
            "observation_interpretation",
            question,
            explanation,
            row,
        )
    )

# ============================================================
# 7. TASK 3 — STELLAR / ORBITAL COMPARISON
# ============================================================

comparison_df = df[
    df["st_mass"].notna() &
    df["pl_orbper"].notna()
].copy()

for _, row in comparison_df.sample(
    min(len(comparison_df), 3000),
    random_state=42,
).iterrows():

    mass = float(row["st_mass"])
    period = float(row["pl_orbper"])

    if mass > 1.2:
        interpretation = (
            "The host star is more massive than the Sun. "
            "For a fixed orbital distance, a more massive star "
            "produces a stronger gravitational field and therefore "
            "generally corresponds to a shorter orbital period."
        )
    elif mass < 0.8:
        interpretation = (
            "The host star is less massive than the Sun. "
            "Its weaker gravitational field changes the relationship "
            "between orbital distance and orbital period."
        )
    else:
        interpretation = (
            "The host star has a mass broadly comparable to the Sun, "
            "so the orbital dynamics are on a roughly solar-like scale."
        )

    question = (
        f"{row['pl_name']} has an orbital period of "
        f"{period:.3f} days around a star with mass "
        f"{mass:.3f} solar masses. "
        f"What can the stellar mass tell us about the system's "
        f"orbital dynamics?"
    )

    examples.append(
        make_example(
            "stellar_dynamics",
            question,
            interpretation,
            row,
        )
    )

# ============================================================
# 8. TASK 4 — PLANET PROPERTY INTERPRETATION
# ============================================================

property_df = df[
    df["pl_masse"].notna() &
    df["pl_rade"].notna() &
    (df["pl_rade"] > 0)
].copy()

for _, row in property_df.iterrows():

    mass = float(row["pl_masse"])
    radius = float(row["pl_rade"])
    density_ratio = float(row["density_earth_ratio"])

    question = (
        f"{row['pl_name']} has a measured mass of "
        f"{mass:.2f} Earth masses and radius of "
        f"{radius:.2f} Earth radii. "
        f"What does the mass-to-radius relationship tell us "
        f"about its approximate bulk density?"
    )

    answer = (
        f"Its mass-to-radius ratio implies an approximate density "
        f"of {density_ratio:.2f} times Earth's density. "
        f"This is a bulk-density estimate derived from the measured "
        f"mass and radius; it does not by itself determine the "
        f"planet's detailed internal composition."
    )

    examples.append(
        make_example(
            "planet_property_reasoning",
            question,
            answer,
            row,
        )
    )

# ============================================================
# 9. REMOVE DUPLICATES
# ============================================================

unique = {}

for example in examples:

    question = example["messages"][0]["content"]

    key = hashlib.sha256(
        question.encode("utf-8")
    ).hexdigest()

    unique[key] = example

examples = list(unique.values())

print("Generated examples:", len(examples))

# ============================================================
# 10. SHUFFLE
# ============================================================

rng = np.random.default_rng(42)

indices = np.arange(len(examples))
rng.shuffle(indices)

examples = [
    examples[i]
    for i in indices
]

# ============================================================
# 11. SPLIT BY HOST STAR
# ============================================================

hosts = sorted(
    {
        x["metadata"]["host_star"]
        for x in examples
    }
)

rng = np.random.default_rng(42)
rng.shuffle(hosts)

n = len(hosts)

train_hosts = set(hosts[:int(n * 0.80)])
val_hosts = set(
    hosts[
        int(n * 0.80):
        int(n * 0.90)
    ]
)
test_hosts = set(
    hosts[int(n * 0.90):]
)

train = []
validation = []
test = []

for example in examples:

    host = example["metadata"]["host_star"]

    if host in train_hosts:
        train.append(example)

    elif host in val_hosts:
        validation.append(example)

    else:
        test.append(example)

# ============================================================
# 12. SAVE JSONL
# ============================================================

def save_jsonl(path, data):

    with open(path, "w", encoding="utf-8") as f:

        for item in data:

            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                )
                + "\n"
            )


save_jsonl(
    OUT / "train.jsonl",
    train,
)

save_jsonl(
    OUT / "validation.jsonl",
    validation,
)

save_jsonl(
    OUT / "test.jsonl",
    test,
)

# ============================================================
# 13. SAVE RAW DATA
# ============================================================

df.to_csv(
    OUT / "source_exoplanets.csv",
    index=False,
)

# ============================================================
# 14. PROVENANCE
# ============================================================

provenance = {
    "dataset": "Astronomy Fine-Tuning Dataset v1",
    "source": "NASA Exoplanet Archive",
    "source_table": "ps",
    "api": API_URL,
    "query": QUERY.strip(),
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "random_seed": 42,
    "raw_rows": int(len(raw)),
    "clean_rows": int(len(df)),
    "examples": int(len(examples)),
    "train_examples": int(len(train)),
    "validation_examples": int(len(validation)),
    "test_examples": int(len(test)),
    "host_stars": int(len(hosts)),
    "tasks": sorted(
        {
            x["metadata"]["task"]
            for x in examples
        }
    ),
}

with open(
    OUT / "provenance.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        provenance,
        f,
        indent=2,
    )

# ============================================================
# 15. DATASET CARD
# ============================================================

dataset_card = f"""# Astronomy Fine-Tuning Dataset v1

## Source

NASA Exoplanet Archive

Table: `ps`

API:
{API_URL}

## Purpose

Domain-specific instruction fine-tuning for astronomy and
scientific reasoning.

## Source statistics

- Raw records: {len(raw)}
- Clean records: {len(df)}
- Generated examples: {len(examples)}
- Training examples: {len(train)}
- Validation examples: {len(validation)}
- Test examples: {len(test)}
- Host stars: {len(hosts)}

## Tasks

- Orbital reasoning
- Observation interpretation
- Stellar dynamics
- Planet property reasoning

## Data integrity

The train, validation and test sets are separated by host star
rather than randomly splitting individual planets. This reduces
the possibility of related planets from the same stellar system
appearing across multiple splits.

## Important limitation

Generated instruction answers are based on deterministic
calculations and rule-based scientific interpretations. They
should be evaluated against the underlying NASA measurements and
not treated as independent scientific literature.
"""

(OUT / "dataset_card.md").write_text(
    dataset_card,
    encoding="utf-8",
)

print()
print("=" * 70)
print("DATASET BUILD COMPLETE")
print("=" * 70)
print("Output:", OUT)
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print()
print("Tasks:")

for task in sorted(
    {
        x["metadata"]["task"]
        for x in examples
    }
):
    print(" -", task)

Downloaded: 32483 rows
Generated examples: 15845

DATASET BUILD COMPLETE
Output: fine_tuning_lab/data/astronomy_ft_v1
Train: 12711
Validation: 1552
Test: 1582

Tasks:
 - observation_interpretation
 - orbital_reasoning
 - planet_property_reasoning
 - stellar_dynamics


In [16]:
from datasets import load_dataset

DATA_DIR = "fine_tuning_lab/data/astronomy_ft_v1"

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/validation.jsonl",
        "test": f"{DATA_DIR}/test.jsonl",
    },
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 12711
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 1552
    })
    test: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 1582
    })
})


In [17]:
from collections import Counter

for split in ["train", "validation", "test"]:
    counts = Counter(
        x["metadata"]["task"]
        for x in dataset[split]
    )

    print("\n", split)
    for task, count in counts.items():
        print(f"{task}: {count}")


 train
observation_interpretation: 4725
orbital_reasoning: 4725
planet_property_reasoning: 863
stellar_dynamics: 2398

 validation
orbital_reasoning: 571
observation_interpretation: 571
stellar_dynamics: 308
planet_property_reasoning: 102

 test
orbital_reasoning: 581
stellar_dynamics: 294
observation_interpretation: 581
planet_property_reasoning: 126


In [18]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded")
print("Device:", model.device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded
Device: cuda:0


In [19]:
example = dataset["test"][0]

messages = example["messages"]

prompt = tokenizer.apply_chat_template(
    messages[:1],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.2,
        do_sample=True,
        top_p=0.9,
    )

generated = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("QUESTION:")
print(messages[0]["content"])

print("\nQWEN BASELINE:")
print(generated)

print("\nREFERENCE:")
print(messages[1]["content"])

QUESTION:
K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

QWEN BASELINE:
To find the distance of the planet from its star using Kepler's Third Law, we can use the following form of the law:

\[ T^2 = \frac{4\pi^2}{G(M_1 + M_2)} a^3 \]

Where:
- \( T \) is the orbital period of the planet.
- \( G \) is the gravitational constant.
- \( M_1 \) and \( M_2 \) are the masses of the two bodies (in this case, the planet and the star).
- \( a \) is the semi-major axis of the orbit (the average distance

REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.


In [20]:
import json
import re
import time
from pathlib import Path

import torch
from tqdm.auto import tqdm

BASELINE_DIR = Path("fine_tuning_lab/baseline")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 100

test_subset = dataset["test"].select(
    range(min(N_SAMPLES, len(dataset["test"])))
)

results = []

for example in tqdm(test_subset):

    question = example["messages"][0]["content"]
    reference = example["messages"][1]["content"]

    prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "user",
                "content": question,
            }
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    start = time.perf_counter()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    elapsed = time.perf_counter() - start

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    results.append({
        "question": question,
        "reference": reference,
        "prediction": generated,
        "task": example["metadata"]["task"],
        "planet": example["metadata"]["planet"],
        "host_star": example["metadata"]["host_star"],
        "generation_seconds": elapsed,
        "generated_tokens": int(
            output.shape[1] -
            inputs["input_ids"].shape[1]
        ),
    })

with open(
    BASELINE_DIR / "qwen7b_baseline_100.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", BASELINE_DIR / "qwen7b_baseline_100.json")
print("Examples:", len(results))

  0%|          | 0/100 [00:00<?, ?it/s]

Saved: fine_tuning_lab/baseline/qwen7b_baseline_100.json
Examples: 100


In [21]:
for i, r in enumerate(results[:5]):

    print("=" * 100)
    print("EXAMPLE", i + 1)
    print("=" * 100)

    print("\nQUESTION:")
    print(r["question"])

    print("\nQWEN:")
    print(r["prediction"])

    print("\nREFERENCE:")
    print(r["reference"])

    print("\nTASK:")
    print(r["task"])


print("Baseline examples:", len(results))

avg_time = sum(
    r["generation_seconds"]
    for r in results
) / len(results)

avg_tokens = sum(
    r["generated_tokens"]
    for r in results
) / len(results)

print(f"Average generation time: {avg_time:.2f}s")
print(f"Average generated tokens: {avg_tokens:.1f}")

EXAMPLE 1

QUESTION:
K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

QWEN:
To find the distance of the planet from its star using Kepler's Third Law, we can use the following form of the law:

\[ T^2 = \frac{4\pi^2}{G(M_1 + M_2)} a^3 \]

Where:
- \( T \) is the orbital period,
- \( G \) is the gravitational constant,
- \( M_1 \) and \( M_2 \) are the masses of the two bodies (in this case, the planet and the star),
- \( a \) is the semi-major axis of the orbit (which we want to find).



REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.

TASK:
orbital_reasoning
EXAMPLE 2

QUESTION:
Kepler-289 d has an orbital period of 66.028 days around a star with mass 1.080 solar masses. What can the stellar mass tell us about the s

In [22]:
import re
import json
import numpy as np
from pathlib import Path

baseline_path = Path(
    "fine_tuning_lab/baseline/qwen7b_baseline_100.json"
)

with open(baseline_path, encoding="utf-8") as f:
    baseline = json.load(f)


def extract_au(text):
    matches = re.findall(
        r"([-+]?\d*\.?\d+)\s*(?:AU|au)\b",
        text,
    )

    if not matches:
        return None

    return float(matches[-1])


def extract_reference_au(text):
    return extract_au(text)


def numerical_accuracy(row):
    predicted = extract_au(row["prediction"])
    expected = extract_reference_au(row["reference"])

    if predicted is None or expected is None:
        return {
            "predicted": predicted,
            "expected": expected,
            "relative_error": None,
            "within_5pct": False,
            "within_10pct": False,
            "within_20pct": False,
        }

    error = abs(predicted - expected) / max(
        abs(expected),
        1e-12,
    )

    return {
        "predicted": predicted,
        "expected": expected,
        "relative_error": error,
        "within_5pct": error <= 0.05,
        "within_10pct": error <= 0.10,
        "within_20pct": error <= 0.20,
    }


scored = []

for row in baseline:

    score = numerical_accuracy(row)

    scored.append({
        **row,
        "numerical": score,
        "completed": (
            score["predicted"] is not None
            or row["task"] != "orbital_reasoning"
        ),
    })


orbital = [
    x for x in scored
    if x["task"] == "orbital_reasoning"
]

print("=" * 70)
print("QWEN 7B BASELINE")
print("=" * 70)

print("Total examples:", len(scored))
print("Orbital examples:", len(orbital))

if orbital:

    print(
        "Numerical answer extracted:",
        sum(
            x["numerical"]["predicted"] is not None
            for x in orbital
        ) / len(orbital)
    )

    print(
        "Within 5%:",
        sum(
            x["numerical"]["within_5pct"]
            for x in orbital
        ) / len(orbital)
    )

    print(
        "Within 10%:",
        sum(
            x["numerical"]["within_10pct"]
            for x in orbital
        ) / len(orbital)
    )

    print(
        "Within 20%:",
        sum(
            x["numerical"]["within_20pct"]
            for x in orbital
        ) / len(orbital)
    )

print()

for task in sorted(
    set(x["task"] for x in scored)
):

    subset = [
        x for x in scored
        if x["task"] == task
    ]

    avg_tokens = np.mean([
        x["generated_tokens"]
        for x in subset
    ])

    avg_seconds = np.mean([
        x["generation_seconds"]
        for x in subset
    ])

    print(
        f"{task:30s}"
        f" n={len(subset):3d}"
        f" tokens={avg_tokens:6.1f}"
        f" time={avg_seconds:6.2f}s"
    )


with open(
    "fine_tuning_lab/baseline/qwen7b_baseline_scored_100.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        scored,
        f,
        indent=2,
        ensure_ascii=False,
    )

print()
print("Saved scored baseline.")

QWEN 7B BASELINE
Total examples: 100
Orbital examples: 42
Numerical answer extracted: 0.0
Within 5%: 0.0
Within 10%: 0.0
Within 20%: 0.0

observation_interpretation     n= 33 tokens= 128.0 time= 10.02s
orbital_reasoning              n= 42 tokens= 128.0 time= 10.01s
planet_property_reasoning      n=  4 tokens= 128.0 time= 10.03s
stellar_dynamics               n= 21 tokens= 128.0 time=  9.96s

Saved scored baseline.


In [23]:
orbital_failures = [
    x for x in scored
    if (
        x["task"] == "orbital_reasoning"
        and not x["numerical"]["within_20pct"]
    )
]

print(
    "Orbital failures:",
    len(orbital_failures),
)

for x in orbital_failures[:5]:

    print("=" * 100)
    print("QUESTION:")
    print(x["question"])

    print("\nQWEN:")
    print(x["prediction"])

    print("\nREFERENCE:")
    print(x["reference"])

    print("\nNUMERICAL:")
    print(x["numerical"])

Orbital failures: 42
QUESTION:
K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

QWEN:
To find the distance of the planet from its star using Kepler's Third Law, we can use the following form of the law:

\[ T^2 = \frac{4\pi^2}{G(M_1 + M_2)} a^3 \]

Where:
- \( T \) is the orbital period,
- \( G \) is the gravitational constant,
- \( M_1 \) and \( M_2 \) are the masses of the two bodies (in this case, the planet and the star),
- \( a \) is the semi-major axis of the orbit (which we want to find).



REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.

NUMERICAL:
{'predicted': None, 'expected': 0.062, 'relative_error': None, 'within_5pct': False, 'within_10pct': False, 'within_20pct': False}
QUESTION:
Kepler-328 c orbits Ke

In [24]:
print("Baseline examples:", len(results))

print(
    "Average generation time:",
    sum(r["generation_seconds"] for r in results)
    / len(results)
)

print(
    "Average generated tokens:",
    sum(r["generated_tokens"] for r in results)
    / len(results)
)

Baseline examples: 100
Average generation time: 10.006280902150001
Average generated tokens: 128.0


In [26]:
from datasets import load_dataset

DATA_DIR = "fine_tuning_lab/data/astronomy_ft_v1"

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/validation.jsonl",
    },
)


def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "text": text
    }


train_dataset = dataset["train"].map(
    format_chat,
    remove_columns=dataset["train"].column_names,
)

val_dataset = dataset["validation"].map(
    format_chat,
    remove_columns=dataset["validation"].column_names,
)

print(train_dataset)
print(val_dataset)

print("\nFIRST TRAIN EXAMPLE")
print(train_dataset[0]["text"])

Map:   0%|          | 0/12711 [00:00<?, ? examples/s]

Map:   0%|          | 0/1552 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 12711
})
Dataset({
    features: ['text'],
    num_rows: 1552
})

FIRST TRAIN EXAMPLE
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
TOI-3464 b was discovered using the Transit method in 2025. What does the discovery method tell us about how this exoplanet was detected?<|im_end|>
<|im_start|>assistant
The transit method detects a planet when it passes in front of its host star from our viewpoint, producing a measurable decrease in the star's observed brightness.<|im_end|>



In [27]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Columns:", train_dataset.column_names)
print("Text type:", type(train_dataset[0]["text"]))
print("Text length:", len(train_dataset[0]["text"]))

Train: 12711
Validation: 1552
Columns: ['text']
Text type: <class 'str'>
Text length: 459


In [28]:
import torch

from peft import LoraConfig
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

print("QLoRA configuration ready")
print(lora_config)

QLoRA configuration ready
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'up_proj', 'down_proj', 'q_proj', 'k_proj', 'o_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_w

In [29]:
SMOKE_SIZE = 500

train_smoke = train_dataset.select(
    range(min(SMOKE_SIZE, len(train_dataset)))
)

print("Smoke training examples:", len(train_smoke))

Smoke training examples: 500


In [30]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model:", MODEL_ID)
print("Loaded:", True)
print("Device:", model.device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model: Qwen/Qwen2.5-7B-Instruct
Loaded: True
Device: cuda:0


In [31]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

model.config.use_cache = False

print("Model prepared for QLoRA")

Model prepared for QLoRA


In [32]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [33]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="fine_tuning_lab/outputs/qwen2.5-7b-qlora-smoke",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    num_train_epochs=1,

    logging_steps=10,
    save_strategy="no",

    fp16=True,
    report_to="none",

    optim="paged_adamw_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_smoke,
)

print("Trainer ready")

Trainer ready


In [34]:
from transformers import DataCollatorForLanguageModeling

MAX_LENGTH = 512

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_smoke.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

tokenized_val = val_dataset.select(
    range(min(100, len(val_dataset)))
).map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(tokenized_train)
print(tokenized_val)
print("Collator ready")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 500
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 100
})
Collator ready


In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Trainer ready")

Trainer ready


In [36]:
batch = data_collator(
    [tokenized_train[i] for i in range(2)]
)

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Batch keys:", batch.keys())

Input shape: torch.Size([2, 130])
Labels shape: torch.Size([2, 130])
Batch keys: KeysView({'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   5207,     40,     12,
             18,     19,     21,     19,    293,    572,  11105,   1667,    279,
          45855,   1714,    304,    220,     17,     15,     17,     20,     13,
           3555,   1558,    279,  18335,   1714,   3291,    601,    911,   1246,
            419,    505,  93460,    295,    572,  16507,     30, 151645,    198,
         151644,  77091,    198,    785,  24065,   1714,  66478,    264,  11580,
            979,    432,  16211,    304,   4065,    315,   1181,   3468,   6774,
            504,   1039,  58385,     11,  17387,    264,  65203,  18472,    304,
            279,   6774,    594,  13166,  32206,     13, 151645,    198, 151643,
     

In [37]:
train_result = trainer.train()

print("SMOKE TRAINING COMPLETE")
print(train_result)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
10,1.338887
20,0.386288
30,0.268868
40,0.264386
50,0.247134
60,0.230107


SMOKE TRAINING COMPLETE
TrainOutput(global_step=63, training_loss=0.44632338342212496, metrics={'train_runtime': 551.3768, 'train_samples_per_second': 0.907, 'train_steps_per_second': 0.114, 'total_flos': 2396716281673728.0, 'train_loss': 0.44632338342212496, 'epoch': 1.0})


In [38]:
ADAPTER_PATH = "fine_tuning_lab/outputs/qwen2.5-7b-qlora-smoke/final_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved:", ADAPTER_PATH)

Adapter saved: fine_tuning_lab/outputs/qwen2.5-7b-qlora-smoke/final_adapter


In [39]:
from pathlib import Path

files = sorted(Path(ADAPTER_PATH).iterdir())

for f in files:
    print(f.name)

print("Adapter files:", len(files))

README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json
Adapter files: 6


In [40]:
import torch

def generate_answer(example, max_new_tokens=128):
    messages = example["messages"]

    prompt = tokenizer.apply_chat_template(
        messages[:1],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return generated

In [42]:
from datasets import load_dataset

DATA_DIR = "fine_tuning_lab/data/astronomy_ft_v1"

test_dataset = load_dataset(
    "json",
    data_files={
        "test": f"{DATA_DIR}/test.jsonl",
    },
)["test"]

print("Test examples:", len(test_dataset))
print("Columns:", test_dataset.column_names)

Generating test split: 0 examples [00:00, ? examples/s]

Test examples: 1582
Columns: ['messages', 'metadata']


In [43]:
for i in range(5):
    example = test_dataset[i]

    print("=" * 100)
    print("QUESTION:")
    print(example["messages"][0]["content"])

    print("\nFINE-TUNED QWEN:")
    print(generate_answer(example))

    print("\nREFERENCE:")
    print(example["messages"][1]["content"])

    print("\nTASK:")
    print(example["metadata"]["task"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


QUESTION:
K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

FINE-TUNED QWEN:


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


The followingYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYou

REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.

TASK:
orbital_reasoning
QUESTION:
Kepler-289 d has an orbital period of 66.028 days around a star with mass 1.080 solar masses. What can the stellar mass tell us about the system's orbital dynamics?

FINE-TUNED QWEN:
The followingYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYouYou

In [44]:
print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("Use cache:", model.config.use_cache)
print("Training mode:", model.training)

Gradient checkpointing: True
Use cache: False
Training mode: True


In [45]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name)
        break

Trainable: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight


In [46]:
with model.disable_adapter():
    model.eval()

    example = test_dataset[0]

    prompt = tokenizer.apply_chat_template(
        example["messages"][:1],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print("BASE MODEL:")
    print(generated)

BASE MODEL:
To find the distance of the planet from its star using Kepler's Third Law, we can use the following form of the law:

\[ T^2 = \frac{4\pi^2}{G(M_1 + M_2)} a^3 \]

Where:
- \( T \) is the orbital


In [47]:
from pathlib import Path

adapter_file = Path(
    "fine_tuning_lab/outputs/qwen2.5-7b-qlora-smoke/final_adapter"
) / "adapter_model.safetensors"

print("Adapter exists:", adapter_file.exists())
print("Adapter size MB:", adapter_file.stat().st_size / 1024**2)

Adapter exists: True
Adapter size MB: 154.05005645751953


again

In [48]:
del model
del trainer

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")


GPU memory cleared


In [49]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Fresh Qwen loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Fresh Qwen loaded


In [50]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False,
)

model.config.use_cache = True

print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("use_cache:", model.config.use_cache)

Gradient checkpointing: False
use_cache: True


In [51]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [52]:
MAX_LENGTH = 512

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_smoke.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

tokenized_val = val_dataset.select(
    range(100)
).map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [53]:
smoke_100 = train_dataset.select(range(100))

tokenized_100 = smoke_100.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print("Smoke examples:", len(tokenized_100))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Smoke examples: 100


In [54]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="fine_tuning_lab/outputs/qwen2.5-7b-qlora-debug",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    num_train_epochs=1,

    logging_steps=5,
    save_strategy="no",

    fp16=True,

    gradient_checkpointing=False,

    optim="paged_adamw_8bit",

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_100,
    data_collator=data_collator,
)

print("Debug trainer ready")

Debug trainer ready


In [55]:
trainer.train()

Step,Training Loss
5,2.396416
10,1.516786


TrainOutput(global_step=13, training_loss=1.773189874795767, metrics={'train_runtime': 85.1593, 'train_samples_per_second': 1.174, 'train_steps_per_second': 0.153, 'total_flos': 483917044660224.0, 'train_loss': 1.773189874795767, 'epoch': 1.0})

In [56]:
model.eval()

example = test_dataset[0]

prompt = tokenizer.apply_chat_template(
    example["messages"][:1],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        use_cache=True,
    )

generated = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("FINE-TUNED:")
print(generated)

print("\nREFERENCE:")
print(example["messages"][1]["content"])

FINE-TUNED:
Using Kepler's Third Law, we can estimate the semi-major axis of the orbit in AU as follows:

\[ a = \left( \frac{P^2 M}{4 \pi^2} \right)^{1/3} \]

Where:
- \( P \) is the orbital period in years

REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.


In [57]:
training_args = TrainingArguments(
    output_dir="fine_tuning_lab/outputs/qwen2.5-7b-qlora",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    num_train_epochs=2,

    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",

    fp16=True,
    gradient_checkpointing=False,

    optim="paged_adamw_8bit",

    report_to="none",

    load_best_model_at_end=True,
)

In [59]:
tokenized_train_full = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(tokenized_train_full)

Map:   0%|          | 0/12711 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 12711
})


In [60]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_full,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

In [61]:
train_result = trainer.train()

print(train_result)

Epoch,Training Loss,Validation Loss
1,0.213586,0.233372
2,0.201184,0.236410


TrainOutput(global_step=3178, training_loss=0.21627695865042934, metrics={'train_runtime': 21176.3805, 'train_samples_per_second': 1.2, 'train_steps_per_second': 0.15, 'total_flos': 1.2178402604747366e+17, 'train_loss': 0.21627695865042934, 'epoch': 2.0})


In [ ]:
print("Train:", len(tokenized_train_full))
print("Validation:", len(tokenized_val))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch:", 
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)

In [63]:
FINAL_ADAPTER = "fine_tuning_lab/outputs/qwen2.5-7b-qlora/final"

trainer.save_model(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)

print("ADAPTER SAVED:", FINAL_ADAPTER)

ADAPTER SAVED: fine_tuning_lab/outputs/qwen2.5-7b-qlora/final


In [64]:
from pathlib import Path

p = Path(FINAL_ADAPTER)

for f in sorted(p.iterdir()):
    print(f.name, f.stat().st_size / 1024**2, "MB")

README.md 0.0049610137939453125 MB
adapter_config.json 0.0011034011840820312 MB
adapter_model.safetensors 154.05005645751953 MB
chat_template.jinja 0.0023908615112304688 MB
tokenizer.json 10.892858505249023 MB
tokenizer_config.json 0.0006618499755859375 MB
training_args.bin 0.004960060119628906 MB


In [65]:
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER = "fine_tuning_lab/outputs/qwen2.5-7b-qlora/final"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER,
)

model.eval()

print("Fine-tuned model loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Fine-tuned model loaded


In [66]:
example = test_dataset[0]

prompt = tokenizer.apply_chat_template(
    example["messages"][:1],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
    )

generated = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("QUESTION:")
print(example["messages"][0]["content"])

print("\nFINE-TUNED QWEN:")
print(generated)

print("\nREFERENCE:")
print(example["messages"][1]["content"])

QUESTION:
K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

FINE-TUNED QWEN:
The estimated semi-major axis is approximately 0.060 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.

REFERENCE:
The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.


In [67]:
for i in range(5):
    example = test_dataset[i]

    prompt = tokenizer.apply_chat_template(
        example["messages"][:1],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            use_cache=True,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    print("=" * 100)
    print("EXAMPLE", i + 1)
    print("=" * 100)
    print("QUESTION:", example["messages"][0]["content"])
    print("\nFINE-TUNED:", generated)
    print("\nREFERENCE:", example["messages"][1]["content"])

EXAMPLE 1
QUESTION: K2-384 e orbits K2-384. The host star has a mass of 0.330 solar masses and the planet has an orbital period of 9.715 days. Using Kepler's third law, approximately how far is the planet from its star in AU?

FINE-TUNED: The estimated semi-major axis is approximately 0.060 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.

REFERENCE: The estimated semi-major axis is approximately 0.062 AU. This estimate uses the stellar mass in solar masses and the orbital period converted to years.
EXAMPLE 2
QUESTION: Kepler-289 d has an orbital period of 66.028 days around a star with mass 1.080 solar masses. What can the stellar mass tell us about the system's orbital dynamics?

FINE-TUNED: The host star has a mass broadly comparable to the Sun, so the orbital dynamics are on a roughly solar-like scale.

REFERENCE: The host star has a mass broadly comparable to the Sun, so the orbital dynamics are on a roughly solar-like scale.
EXAMP

In [68]:
import re
import math
import time

def extract_au(text):
    matches = re.findall(
        r'(?<!\d)(0?\.\d+)\s*(?:AU|au)\b',
        text
    )
    if not matches:
        return None

    return float(matches[-1])


def evaluate_model(model, dataset, name):
    results = []
    start = time.time()

    for i, example in enumerate(dataset):
        prompt = tokenizer.apply_chat_template(
            example["messages"][:1],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                use_cache=True,
            )

        generated = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        reference = example["messages"][1]["content"]

        results.append({
            "task": example["metadata"]["task"],
            "generated": generated,
            "reference": reference,
        })

        if (i + 1) % 100 == 0:
            print(f"{name}: {i + 1}/{len(dataset)}")

    elapsed = time.time() - start

    print(f"\n{name} complete")
    print("Examples:", len(results))
    print("Time:", round(elapsed, 2), "sec")

    return results

In [70]:
fine_results = evaluate_model(
    model,
    test_dataset.select(range(100)),
    "FINE-TUNED",
)

KeyboardInterrupt: 

In [ ]:
orbital_results = [
    r for r in fine_results
    if r["task"] == "orbital_reasoning"
]

correct_5 = 0
correct_10 = 0
correct_20 = 0

for r in orbital_results:
    predicted = extract_au(r["generated"])
    expected = extract_au(r["reference"])

    if predicted is None or expected is None:
        continue

    error = abs(predicted - expected) / expected

    if error <= 0.05:
        correct_5 += 1
    if error <= 0.10:
        correct_10 += 1
    if error <= 0.20:
        correct_20 += 1

print("Orbital examples:", len(orbital_results))
print("Within 5%:", correct_5)
print("Within 10%:", correct_10)
print("Within 20%:", correct_20)

In [71]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Best checkpoint: fine_tuning_lab/outputs/qwen2.5-7b-qlora/checkpoint-1589
Best validation loss: 0.2333715260028839


In [74]:
print("Training complete:", trainer.state.global_step)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Final adapter:", FINAL_ADAPTER)

Training complete: 3178
Best checkpoint: fine_tuning_lab/outputs/qwen2.5-7b-qlora/checkpoint-1589
Final adapter: fine_tuning_lab/outputs/qwen2.5-7b-qlora/final


In [75]:
from pathlib import Path
import shutil

src = Path(FINAL_ADAPTER)
release = Path("fine_tuning_lab/release/astro-qwen2.5-7b-qlora")

release.mkdir(parents=True, exist_ok=True)

for f in src.iterdir():
    if f.is_file():
        shutil.copy2(f, release / f.name)

print("Release files:")
for f in release.iterdir():
    print(f.name, round(f.stat().st_size / 1024**2, 2), "MB")

Release files:
chat_template.jinja 0.0 MB
tokenizer.json 10.89 MB
tokenizer_config.json 0.0 MB
adapter_config.json 0.0 MB
adapter_model.safetensors 154.05 MB
training_args.bin 0.0 MB
README.md 0.0 MB


In [76]:
from pathlib import Path

best_checkpoint = Path(
    "fine_tuning_lab/outputs/qwen2.5-7b-qlora/checkpoint-1589"
)

print("Best checkpoint exists:", best_checkpoint.exists())
print("Files:")
for f in best_checkpoint.iterdir():
    print(f.name)

Best checkpoint exists: True
Files:
chat_template.jinja
tokenizer.json
tokenizer_config.json
adapter_config.json
scheduler.pt
rng_state.pth
adapter_model.safetensors
training_args.bin
README.md
trainer_state.json
optimizer.pt
scaler.pt


In [77]:
trainer._load_from_checkpoint(str(best_checkpoint))

print("BEST CHECKPOINT LOADED")

BEST CHECKPOINT LOADED


In [78]:
best_release = Path(
    "fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best"
)

best_release.mkdir(parents=True, exist_ok=True)

model.save_pretrained(best_release)
tokenizer.save_pretrained(best_release)

print("BEST ADAPTER SAVED")

for f in sorted(best_release.iterdir()):
    print(
        f.name,
        round(f.stat().st_size / 1024**2, 2),
        "MB"
    )

BEST ADAPTER SAVED
README.md 0.0 MB
adapter_config.json 0.0 MB
adapter_model.safetensors 154.05 MB
chat_template.jinja 0.0 MB
tokenizer.json 10.89 MB
tokenizer_config.json 0.0 MB


In [81]:
!pip install -q --upgrade --force-reinstall --no-cache-dir huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 384.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 343.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 303.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 262.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 226.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 332.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 353.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 342.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 199.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 247.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [82]:
import huggingface_hub

print("Hugging Face Hub:", huggingface_hub.__version__)

from huggingface_hub import login, HfApi

print("Hugging Face import: PASS")

Hugging Face Hub: 1.11.0
Error importing huggingface_hub._login: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)


ImportError: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

In [84]:
!pip uninstall -y huggingface_hub
!pip install -q --no-cache-dir "huggingface_hub==0.36.0"

Found existing installation: huggingface_hub 1.29.0
Uninstalling huggingface_hub-1.29.0:
  Successfully uninstalled huggingface_hub-1.29.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 18.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.0 which is incompatible.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.


In [110]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")


In [108]:
!pip install -q -U kaggle

In [109]:
!kaggle models --help

usage: kaggle models [-h]
                     {instances,i,variations,v,get,list,init,create,delete,update,topics}
                     ...

options:
  -h, --help            show this help message and exit

commands:
  {instances,i,variations,v,get,list,init,create,delete,update,topics}
    instances (i, variations, v)
                        Commands related to Kaggle model variations
    get                 Get a model
    list                List models
    init                Initialize metadata file for model creation
    create              Create a new model
    delete              Delete a model
    update              Update a model
    topics              List discussion topics for a model


In [112]:
import os
import kagglehub
from kaggle_secrets import UserSecretsClient

# 1. Load Kaggle credentials
secrets = UserSecretsClient()

os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

print("Kaggle username:", os.environ["KAGGLE_USERNAME"])

# 2. Model handle
username = os.environ["KAGGLE_USERNAME"]

handle = (
    f"{username}/astro-qwen2.5-7b-qlora/"
    f"transformers/v1"
)

# 3. Local adapter directory
model_dir = (
    "fine_tuning_lab/release/"
    "astro-qwen2.5-7b-qlora-best"
)

print("Handle:", handle)
print("Model directory:", model_dir)

# 4. Create model if it doesn't already exist
try:
    kagglehub.model_create(handle)
    print("Model created.")
except Exception as e:
    print("Model already exists or creation skipped:", e)

# 5. Upload adapter
kagglehub.model_upload(
    handle=handle,
    local_model_dir=model_dir,
    version_notes=(
        "Initial release of astronomy-domain QLoRA "
        "adapter for Qwen2.5-7B-Instruct. "
        "Trained on NASA Exoplanet Archive-derived "
        "astronomy reasoning data."
    ),
)

print()
print("==========================================")
print("KAGGLE MODEL PUBLISHED")
print("==========================================")
print(f"https://www.kaggle.com/models/{handle}")

Kaggle username: aashutoshbhardwaj5
Handle: aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1
Model directory: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best
Model already exists or creation skipped: module 'kagglehub' has no attribute 'model_create'
Uploading Model https://kaggle.com/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1 ...
Model 'astro-qwen2.5-7b-qlora' does not exist or access is forbidden for user 'aashutoshbhardwaj5'. Creating or handling Model...
Model 'astro-qwen2.5-7b-qlora' Created.
Starting upload for file fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/chat_template.jinja


Uploading: 100%|██████████| 2.51k/2.51k [00:00<00:00, 3.70kB/s]

Upload successful: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/chat_template.jinja (2KB)
Starting upload for file fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/tokenizer.json



Uploading: 100%|██████████| 11.4M/11.4M [00:00<00:00, 12.7MB/s]

Upload successful: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/tokenizer.json (11MB)
Starting upload for file fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/tokenizer_config.json


...


Uploading: 100%|██████████| 162M/162M [00:02<00:00, 54.8MB/s] 

Upload successful: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/adapter_model.safetensors (154MB)
Starting upload for file fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/README.md



Uploading: 100%|██████████| 5.20k/5.20k [00:00<00:00, 7.78kB/s]

Upload successful: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best/README.md (5KB)


Your model instance has been created.
Files are being processed...
See at: https://kaggle.com/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1

KAGGLE MODEL PUBLISHED
https://www.kaggle.com/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1


In [5]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

# Get Hugging Face token securely
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

# Authenticate
api = HfApi(token=hf_token)

# Your Hugging Face username
hf_username = "aashutoshkumarbhardwaj"

# Public model repository
repo_id = f"{hf_username}/astro-qwen2.5-7b-qlora"

# Our already-trained BEST adapter
model_dir = Path(
    "fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best"
)

print("Repository:", repo_id)
print("Model directory:", model_dir)
print("Directory exists:", model_dir.exists())

print("\nFiles:")
for f in sorted(model_dir.iterdir()):
    print(
        f"  {f.name} "
        f"({f.stat().st_size / 1024**2:.2f} MB)"
    )

HF token loaded: True
Repository: aashutoshkumarbhardwaj/astro-qwen2.5-7b-qlora
Model directory: fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best
Directory exists: False

Files:


FileNotFoundError: [Errno 2] No such file or directory: 'fine_tuning_lab/release/astro-qwen2.5-7b-qlora-best'

In [2]:
!pip install -q -U kagglehub huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 18.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [3]:
import kagglehub

model_path = kagglehub.model_download(
    "aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1"
)

print("Downloaded to:")
print(model_path)

Downloaded to:
/kaggle/input/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1/1


In [6]:
from pathlib import Path

path = Path(model_path)

for f in sorted(path.rglob("*")):
    if f.is_file():
        print(
            f.relative_to(path),
            round(f.stat().st_size / 1024**2, 2),
            "MB"
        )

README.md 0.0 MB
adapter_config.json 0.0 MB
adapter_model.safetensors 154.05 MB
chat_template.jinja 0.0 MB
tokenizer.json 10.89 MB
tokenizer_config.json 0.0 MB


In [8]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

# Get HF token securely
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# Your HF repo
repo_id = "aashutoshkumarbhardwaj/astro-qwen2.5-7b-qlora"

api = HfApi(token=hf_token)

# Create repo
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
)

# Upload downloaded Kaggle model
api.upload_folder(
    folder_path=model_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Initial release of Astro Qwen2.5 7B QLoRA",
)

print("======================================")
print("HUGGING FACE UPLOAD COMPLETE")
print("======================================")
print(f"https://huggingface.co/{repo_id}")

ImportError: cannot import name 'XetProgressReporter' from 'huggingface_hub.utils._xet_progress_reporting' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_xet_progress_reporting.py)

In [9]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("Xet disabled")

Xet disabled


In [10]:
!pip uninstall -y huggingface_hub
!pip install -q --no-cache-dir huggingface_hub==0.36.0

Found existing installation: huggingface_hub 1.29.0
Uninstalling huggingface_hub-1.29.0:
  Successfully uninstalled huggingface_hub-1.29.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 15.8 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.0 which is incompatible.


In [11]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import HfApi

print("Hugging Face Hub import: PASS")

Hugging Face Hub import: PASS


In [12]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [13]:
import kagglehub

model_path = kagglehub.model_download(
    "aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1"
)

print(model_path)

/kaggle/input/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1/1


In [16]:
from huggingface_hub import HfApi

repo_id = "aashutoshkumarbhardwaj/astro-qwen2.5-7b-qlora"

api = HfApi(token=hf_token)

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
)

api.upload_folder(
    folder_path=model_path,
    repo_id=repo_id,
    repo_type="model",
    path_in_repo=".",
    commit_message="Initial release of Astro Qwen2.5 7B QLoRA",
)

print("======================================")
print("HUGGING FACE UPLOAD COMPLETE")
print("======================================")
print(f"https://huggingface.co/{repo_id}")

ImportError: cannot import name 'XetProgressReporter' from 'huggingface_hub.utils._xet_progress_reporting' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_xet_progress_reporting.py)

In [15]:
!pip uninstall -y hf-xet

Found existing installation: hf-xet 1.6.0
Uninstalling hf-xet-1.6.0:
  Successfully uninstalled hf-xet-1.6.0


In [17]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import HfApi

print("HF Hub:", __import__("huggingface_hub").__version__)

try:
    import hf_xet
    print("ERROR: hf_xet is still installed")
except ImportError:
    print("hf_xet: not installed")

HF Hub: 1.11.0
ERROR: hf_xet is still installed


In [18]:
import kagglehub

model_path = kagglehub.model_download(
    "aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1"
)

print("Model downloaded:")
print(model_path)

Model downloaded:
/kaggle/input/models/aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1/1


In [19]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("Token loaded:", bool(hf_token))

Token loaded: True


In [21]:
!pip install --upgrade huggingface_hub hf_xet


  Using cached huggingface_hub-1.29.0-py3-none-any.whl.metadata (16 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached huggingface_hub-1.29.0-py3-none-any.whl (795 kB)
Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.5 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0


In [22]:
!pip install --no-cache-dir --upgrade huggingface_hub hf_xet


In [25]:
import os
import kagglehub
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# ============================================================
# 1. Download the already-published Kaggle model
# ============================================================

model_path = kagglehub.model_download(
    "aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1"
)

print("Kaggle model downloaded to:")
print(model_path)


# ============================================================
# 2. Get Hugging Face token securely
# ============================================================

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

login(token=hf_token)


# ============================================================
# 3. Set your Hugging Face repository
# ============================================================

repo_id = "aashutoshkumarbhardwaj/astro-qwen2.5-7b-qlora"

api = HfApi(token=hf_token)

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
)

print("Repository ready:", repo_id)


# ============================================================
# 4. Upload every file individually
# ============================================================

for root, dirs, files in os.walk(model_path):

    for file in files:

        local_file_path = os.path.join(root, file)

        relative_path = os.path.relpath(
            local_file_path,
            model_path
        )

        print(f"Uploading: {relative_path}")

        api.upload_file(
            path_or_fileobj=local_file_path,
            path_in_repo=relative_path,
            repo_id=repo_id,
            repo_type="model",
            token=hf_token,
        )


print()
print("==============================================")
print("HUGGING FACE MODEL PUBLISHED")
print("==============================================")
print(f"https://huggingface.co/{repo_id}")

Error importing huggingface_hub._login: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)


ImportError: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

In [24]:
!pip install --no-cache-dir --force-reinstall huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 266.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 291.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 279.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 126.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 307.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 258.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 308.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 386.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 275.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 193.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [26]:
import os
import kagglehub
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# ============================================================
# 1. Download the already-published Kaggle model
# ============================================================

model_path = kagglehub.model_download(
    "aashutoshbhardwaj5/astro-qwen2.5-7b-qlora/transformers/v1"
)

print("Kaggle model downloaded to:")
print(model_path)


# ============================================================
# 2. Get Hugging Face token securely
# ============================================================

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

login(token=hf_token)


# ============================================================
# 3. Set your Hugging Face repository
# ============================================================

repo_id = "aashutoshkumarbhardwaj/astro-qwen2.5-7b-qlora"

api = HfApi(token=hf_token)

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
)

print("Repository ready:", repo_id)


# ============================================================
# 4. Upload every file individually
# ============================================================

for root, dirs, files in os.walk(model_path):

    for file in files:

        local_file_path = os.path.join(root, file)

        relative_path = os.path.relpath(
            local_file_path,
            model_path
        )

        print(f"Uploading: {relative_path}")

        api.upload_file(
            path_or_fileobj=local_file_path,
            path_in_repo=relative_path,
            repo_id=repo_id,
            repo_type="model",
            token=hf_token,
        )


print()
print("==============================================")
print("HUGGING FACE MODEL PUBLISHED")
print("==============================================")
print(f"https://huggingface.co/{repo_id}")

Error importing huggingface_hub._login: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)


ImportError: cannot import name 'DeviceCodeError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)